## 1. Aplicación de técnicas de aprendizaje no supervisado

En esta fase se busca descubrir patrones en el dataset Energy Efficiency mediante algoritmos de clustering.

El objetivo es segmentar los edificios en grupos con comportamiento energético similar y generar un dataset etiquetado a partir de los resultados obtenidos.

## 1. Objetivo

Aplicar técnicas de aprendizaje no supervisado para descubrir patrones en el dataset Energy Efficiency.

Se busca segmentar los edificios en grupos homogéneos, justificar el número de clusters elegido, interpretar los perfiles obtenidos y generar un dataset etiquetado a partir del clustering.

## 2. Carga y revisión del dataset

Se reutiliza el archivo procesado generado en la fase 1, donde las variables de entrada ya fueron estandarizadas y las variables objetivo se conservaron para interpretar los grupos resultantes.

Esto permite aplicar clustering sin repetir el preprocesamiento básico.

In [10]:
import pandas as pd

import seaborn as sns


sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/data_clean.csv")
X = df.drop(columns=["Y1", "Y2"])

print("Dimensión del dataset:", df.shape)
print("Columnas:", list(df.columns))
print("Valores faltantes por columna:")
print(df.isnull().sum())
print("\nResumen descriptivo de las variables de entrada:")
display(X.describe().T)

df.head()

Dimensión del dataset: (768, 10)
Columnas: ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'Y1', 'Y2']
Valores faltantes por columna:
X1    0
X2    0
X3    0
X4    0
X5    0
X6    0
X7    0
X8    0
Y1    0
Y2    0
dtype: int64

Resumen descriptivo de las variables de entrada:


,count,mean,std,min,25%,50%,75%,max
X1,768.0,4.625929e-18,1.000652,-1.363361,-0.771796,-0.132905,0.529647,2.044054
X2,768.0,-4.718448e-16,1.000652,-1.785875,-0.742182,0.023193,0.788568,1.553943
X3,768.0,5.181041e-16,1.000652,-1.686934,-0.562800,-0.000733,0.561334,2.247536
X4,768.0,3.238150e-16,1.000652,-1.470077,-0.791580,0.158316,0.972512,0.972512
X5,768.0,0.000000e+00,1.000652,-1.000000,-1.000000,0.000000,1.000000,1.000000
X6,768.0,0.000000e+00,1.000652,-1.341641,-0.670820,0.000000,0.670820,1.341641
X7,768.0,1.480297e-16,1.000652,-1.762934,-1.011311,0.116124,1.243559,1.243559
X8,768.0,3.700743e-17,1.000652,-1.814575,-0.685506,0.120972,0.766154,1.411336


,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,2.044054,-1.785875,-0.562800,-1.470077,1.0,-1.341641,-1.762934,-1.814575,15.55,21.33
1,2.044054,-1.785875,-0.562800,-1.470077,1.0,-0.447214,-1.762934,-1.814575,15.55,21.33
2,2.044054,-1.785875,-0.562800,-1.470077,1.0,0.447214,-1.762934,-1.814575,15.55,21.33
3,2.044054,-1.785875,-0.562800,-1.470077,1.0,1.341641,-1.762934,-1.814575,15.55,21.33
4,1.286851,-1.229239,-0.000733,-1.198678,1.0,-1.341641,-1.762934,-1.814575,20.84,28.28


## 3. Estrategia de clustering

Se utilizará K-means porque el dataset ya fue estandarizado en la fase 1 y el algoritmo funciona bien sobre variables numéricas con escalas comparables.

Para justificar el número de clusters se evaluarán valores de k entre 2 y 6 usando inercia y silhouette score. Se seleccionará el k que ofrezca la mejor separación entre grupos.

In [11]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_values = range(2, 7)
metrics = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    metrics.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X, labels),
        }
    )

metrics_df = pd.DataFrame(metrics)
metrics_df

,k,inertia,silhouette
0,2,3497.829376,0.390531
1,3,3085.179128,0.294440
2,4,2699.047546,0.219831
3,5,2455.184957,0.218544
4,6,2263.624620,0.203838


## 4. Ajuste final del modelo de clustering

Con base en la comparación de métricas, se selecciona el valor de k con mejor separación entre grupos para entrenar el modelo final de K-means.

In [12]:
final_k = 2
kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)

df_clustered = df.copy()
df_clustered["cluster"] = kmeans.fit_predict(X)

cluster_summary = df_clustered.groupby("cluster")[["Y1", "Y2"]].mean()
cluster_summary["energy_score"] = cluster_summary["Y1"] + cluster_summary["Y2"]

low_cluster = cluster_summary["energy_score"].idxmin()
high_cluster = cluster_summary["energy_score"].idxmax()

cluster_labels = {
    low_cluster: "Baja demanda energética",
    high_cluster: "Alta demanda energética",
}

df_clustered["cluster_label"] = df_clustered["cluster"].map(cluster_labels)

cluster_summary.sort_values("energy_score")

,Y1,Y2,energy_score
cluster,,,
0,13.338505,16.071432,29.409937
1,31.275885,33.104089,64.379974


In [13]:
cluster_profile = df_clustered.groupby("cluster").agg(
    count=("cluster", "size"),
    x1_mean=("X1", "mean"),
    x2_mean=("X2", "mean"),
    x3_mean=("X3", "mean"),
    x4_mean=("X4", "mean"),
    x5_mean=("X5", "mean"),
    x6_mean=("X6", "mean"),
    x7_mean=("X7", "mean"),
    x8_mean=("X8", "mean"),
    y1_mean=("Y1", "mean"),
    y2_mean=("Y2", "mean"),
)

cluster_profile["energy_score"] = cluster_profile["y1_mean"] + cluster_profile["y2_mean"]
cluster_profile.sort_values("energy_score")

,count,x1_mean,x2_mean,x3_mean,x4_mean,x5_mean,x6_mean,x7_mean,x8_mean,y1_mean,y2_mean,energy_score
cluster,,,,,,,,,,,,
0,384,-0.827009,0.858148,-0.281766,0.972512,-1.0,0.0,-0.001317,7.517135e-18,13.338505,16.071432,29.409937
1,384,0.827009,-0.858148,0.281766,-0.972512,1.0,0.0,0.001317,7.517135e-18,31.275885,33.104089,64.379974


## 5. Interpretación de los clusters

A partir de la media de las variables objetivo Y1 y Y2 se asignan etiquetas descriptivas a los grupos obtenidos.

El cluster 0 presenta la menor demanda energética promedio, con valores más bajos de Y1 y Y2, y agrupa 384 registros. Este grupo se etiqueta como "Baja demanda energética".

El cluster 1 presenta una demanda energética significativamente mayor, también con 384 registros, y se etiqueta como "Alta demanda energética".

En términos de variables de entrada, el grupo de baja demanda se asocia con valores estandarizados más bajos en X5 y más altos en X2 y X4, mientras que el grupo de alta demanda muestra el patrón contrario.

In [14]:
output_path = "../data/processed/data_clustered.csv"
df_clustered.to_csv(output_path, index=False)

print(f"Dataset etiquetado exportado en: {output_path}")
df_clustered[["cluster", "cluster_label"]].head()

Dataset etiquetado exportado en: ../data/processed/data_clustered.csv


,cluster,cluster_label
0,1,Alta demanda energética
1,1,Alta demanda energética
2,1,Alta demanda energética
3,1,Alta demanda energética
4,1,Alta demanda energética
